# Silver V2 — Safe boilerplate cleanup

Uses existing verified **Silver V1** on Google Drive. Dry-run first, then deliberately enable publishing. Does not touch Gold V1, and does not download Bronze.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = "feat/silver-v2-boilerplate-cleaning"
PROJECT_DIR = Path("/content/project")
if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError("The project path contains non-git files")
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                    REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=PROJECT_DIR, check=True)
print("Branch:", BRANCH)


In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)
DATA_LAKE_ROOT = Path(MOUNT) / "MyDrive" / "vietnamese-multilabel-topic-classification"
SILVER_V1_REPORT = "silver_quality_v1_b81fcce589_seed42_buf10000_n50000.json"
if not (DATA_LAKE_ROOT / "05_reports" / SILVER_V1_REPORT).is_file():
    raise FileNotFoundError("Mount the Drive account containing the Silver V1 report")
print(DATA_LAKE_ROOT)


In [ ]:
import json
import os
from vietnamese_topic_classification.data.silver.reader import SilverReader
from vietnamese_topic_classification.data.silver.pipeline_v2 import SilverV2Pipeline

reader = SilverReader(DATA_LAKE_ROOT, SILVER_V1_REPORT)
verified = reader.verify()
pipeline = SilverV2Pipeline(reader)
expected = pipeline.planned_paths(verified["sha256"])
print("Verified Silver V1:", verified)
print("Silver V2 destinations:", {k: str(v) for k, v in expected.items()})
print("Existing:", [str(p) for p in expected.values() if p.exists()])


In [ ]:
# Dry-run checks all records but does NOT create Drive outputs.
result = pipeline.run(dry_run=True)
print(json.dumps(result["report"], ensure_ascii=False, indent=2))
print("Review changed/review/guardrail counts before choosing PUBLISH.")


In [ ]:
# Deliberate opt-in. Keep False until you have reviewed dry-run totals.
PUBLISH = False
if PUBLISH:
    result = pipeline.run()
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print("Dry-run only; no Silver V2 output published. Set PUBLISH=True to create Silver V2.")


In [ ]:
# Read-only verification, even when Silver V2 was published in another session.
from vietnamese_topic_classification.data.silver.reader import SilverReader
if expected["report"].is_file():
    v2_reader = SilverReader(DATA_LAKE_ROOT, expected["report"].name)
    print("Silver V2:", v2_reader.verify())
    quality = json.loads(expected["report"].read_text(encoding="utf-8"))
    print("Cleaning:", quality["cleaned_rows"], "Review:", quality["review_rows"],
          "Guardrail:", quality["guardrail_rows"])
    print("Audit:", expected["audit"])
else:
    print("Silver V2 not published yet.")
